In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from agent.components.GaussianProcess import GASK
from agent.components.commons import ServiceType

df = pd.read_csv("../statics/metrics_20_0.csv")
# 2. Initialize and train
rask_gp = GASK(show_figures=False)
rask_gp.init_models(df, density=1.0)

INFO:GP_Model:Fitting GP for elastic-workbench-qr-detector - Target: max_tp
INFO:multiscale:train_gp_models took 662 ms to execute


In [2]:
import numpy as np
%load_ext autoreload
%autoreload 2

rask_gp.predict(ServiceType.QR, "max_tp", {'data_quality': 100, 'cores': 6.0})

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


(np.float64(2050.112908336468), np.float64(145.8892622660747))

In [22]:
from agent.components.GaussianProcess import get_empirical_boundaries
%load_ext autoreload
%autoreload 2

from agent.components.Optimizer import local_obj, solve_global
from agent.components.SLORegistry_v2 import SLO_Registry

slo_lib = SLO_Registry("../statics/config/service_level_objectives.yml")
slos = slo_lib.get_slo_for_client("experiment-1", "client-1")

# empirical_bounds = get_empirical_boundaries(rask_gp)[ServiceType.QR]
# print(empirical_bounds)
# x = normalize_value_in_bounds({'cores': 1.0, 'data_quality': 990}, empirical_bounds)
x = [1.0, 120]

# local_obj(x, slos, rask_gp)
solve_global(slos, rask_gp, last_assignments=x)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Calculated SLO-F for {'cores': np.float64(1.0), 'data_quality': np.float64(120.0)}: 0.16660971946718695
Calculated SLO-F for {'cores': np.float64(1.000000074505806), 'data_quality': np.float64(120.0)}: 0.16660972985574773
Calculated SLO-F for {'cores': np.float64(1.0), 'data_quality': np.float64(120.00001466274261)}: 0.16660969887596327
Calculated SLO-F for {'cores': np.float64(4.485822565853596), 'data_quality': np.float64(6.0)}: 0.7205636407719382
Calculated SLO-F for {'cores': np.float64(4.485822640359402), 'data_quality': np.float64(6.0)}: 0.7205636478006621
Calculated SLO-F for {'cores': np.float64(4.485822565853596), 'data_quality': np.float64(6.000014662742615)}: 0.7205636408790008
Calculated SLO-F for {'cores': np.float64(5.999999999999991), 'data_quality': np.float64(41.84096486832783)}: 0.8010415304757308
Calculated SLO-F for {'cores': np.float64(5.999999925494185), 'data_quality': np.floa

[np.float64(5.943052671874339), np.float64(136.74849055333002)]

In [10]:
import numpy as np

# Convert to a numpy array so we can do math on the whole vector
x_norm = np.array([0.1, 0.1])
raw_bounds = get_empirical_boundaries(rask_gp.training_data)[ServiceType.QR]
del raw_bounds['max_tp']

# Store these to use for de-normalization inside the objective
ordered_bounds = list(raw_bounds.values())

for e in [1e-5, 1e-3, 1e-2, 5e-2]:
    # val_start uses the original center
    val_start = local_obj(x_norm, slos, rask_gp, ordered_bounds)

    # x_norm + e now adds 'e' to every element (e.g., [0.11, 0.11])
    val_nudge = local_obj(x_norm + e, slos, rask_gp, ordered_bounds)

    diff = abs(val_start - val_nudge)
    print(f"Eps {e}: Change in SLO-F is {diff:.6f}")

Calculated SLO-F for {'cores': np.float64(1.5), 'data_quality': np.float64(104.4)}: 0.25936059126189825
Calculated SLO-F for {'cores': np.float64(1.5000499999999999), 'data_quality': np.float64(104.40984)}: 0.2593529338264404
Eps 1e-05: Change in SLO-F is 0.000008
Calculated SLO-F for {'cores': np.float64(1.5), 'data_quality': np.float64(104.4)}: 0.25936059126189825
Calculated SLO-F for {'cores': np.float64(1.505), 'data_quality': np.float64(105.384)}: 0.25859717886572176
Eps 0.001: Change in SLO-F is 0.000763
Calculated SLO-F for {'cores': np.float64(1.5), 'data_quality': np.float64(104.4)}: 0.25936059126189825
Calculated SLO-F for {'cores': np.float64(1.55), 'data_quality': np.float64(114.24)}: 0.2519625606278182
Eps 0.01: Change in SLO-F is 0.007398
Calculated SLO-F for {'cores': np.float64(1.5), 'data_quality': np.float64(104.4)}: 0.25936059126189825
Calculated SLO-F for {'cores': np.float64(1.75), 'data_quality': np.float64(153.60000000000002)}: 0.22958445201832267
Eps 0.05: Chang